# Data Exploration
This notebook is for me to discover trends in the data we were given.


In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
import numpy as np
import plotly.graph_objects as go
import seaborn as sns
from datetime import time
from plotly.subplots import make_subplots

In [7]:
dt_train = pd.read_csv("../data/raw/train_selected.csv")
dt_test = pd.read_csv("../data/raw/test_selected.csv")
loc = dt_train.loc[180]
loc
#dt_test.head()

id              1.00
cycle         181.00
s1           1420.64
s2            551.83
s3             48.10
s4            520.43
ttf            11.00
label_bnc       1.00
Name: 180, dtype: float64

In [4]:
# Distribution de la durée de vie maximale des moteurs (TTF)
ttf_max = dt_train.groupby('id')['ttf'].max().reset_index()
fig_hist = px.histogram(ttf_max, x='ttf', nbins=20, 
                        title='Distribution of engines maximal life (in engine cycles)',
                        labels={'ttf': 'Numbers of cycles before failure'})
fig_hist.show()
#fig_hist.write_html("../figures/ttf_distri.html")
fig_hist.write_image("../figures/ttf_distri.png")
# Évolution des capteurs pour le Moteur 1
moteur_1 = dt_train[dt_train['id'] == 1]

# Transformation des données au format long pour faciliter les sous-graphiques avec Plotly Express
moteur_1_melted = moteur_1.melt(id_vars=['cycle', 'ttf'], 
                                value_vars=['s1', 's2', 's3', 's4'], 
                                var_name='Sensor', 
                                value_name='Unit')

# Création des sous-graphiques par capteur
fig_line = px.line(moteur_1_melted, x='cycle', y='Unit', facet_row='Sensor',
                   title='Sensors evolution (engine 1)', 
                   height=800)

# Désolidariser les axes Y pour que chaque capteur ait sa propre échelle
fig_line.update_yaxes(matches=None) 
fig_line.show()

In [8]:
dt_train['Condition'] = dt_train['label_bnc'].map({0: 'Healthy', 1: 'Faulty'})

# Melt the dataframe to have all sensors in a single column for facet_row
df_melted = dt_train.melt(id_vars=['id', 'cycle', 'Condition'], 
                          value_vars=['s1', 's2', 's3', 's4'], 
                          var_name='Sensor', 
                          value_name='Measurement')

# Create the faceted line chart
fig_all_engines = px.line(df_melted, 
                          x='cycle', 
                          y='Measurement', 
                          color='Condition', 
                          line_group='id',           # Ensures each engine gets its own connected line
                          facet_row='Sensor',        # Creates 4 stacked graphs
                          color_discrete_map={'Healthy': 'blue', 'Faulty': 'red'}, # Force red for faulty
                          title='Sensor Degradation Across All Engines',
                          height=1000)

# Unlink the y-axes so each sensor scales properly to its own range
fig_all_engines.update_yaxes(matches=None)

# Show the interactive figure
fig_all_engines.show()


In [9]:

fig_all_engines.write_image("../figures/all_engines_sensors.jpeg")

## Understand the correlations

### Pearson correlation
for linear correlation

In [ ]:
corr = dt_train.corr(numeric_only=True)

fig_corr = px.imshow(corr, 
                     text_auto='.2f',  # Displays the correlation values rounded to 2 decimals
                     aspect='auto', 
                     color_continuous_scale='RdBu_r', # RdBu_r is very similar to coolwarm
                     color_continuous_midpoint=0,     # Centers the color scale at 0
                     title='Correlation Heatmap of Engine Data (Pearson)')

# Adjust layout for better readability
fig_corr.update_layout(width=800, height=800)

fig_corr.show()

### Spearman 
for monotonic relations

In [ ]:
corr2 = dt_train.corr(method= "spearman", numeric_only=True)

fig_corr2 = px.imshow(corr2, 
                     text_auto='.2f',  # Displays the correlation values rounded to 2 decimals
                     aspect='auto', 
                     color_continuous_scale='RdBu_r', # RdBu_r is very similar to coolwarm
                     color_continuous_midpoint=0,     # Centers the color scale at 0
                     title='Correlation Heatmap of Engine Data (Spearman)')

# Adjust layout for better readability
fig_corr2.update_layout(width=800, height=800)

fig_corr2.show()

Basically the same heatmap, doesn't change that much, we can follow the same trends from one to another.

In [13]:

fig_corr.write_image("../figures/fig_corr.jpeg")
fig_corr2.write_image("../figures/fig_corr2.jpeg")

In [14]:
corr["ttf"].sort_values()

cycle       -0.736241
s3          -0.696228
s1          -0.678948
label_bnc   -0.566596
id           0.078753
s2           0.657223
s4           0.671983
ttf          1.000000
Name: ttf, dtype: float64

### Distribution of sensors

In [16]:
# Restructuration des données (melt) pour rassembler les 4 capteurs
df_melted_label = dt_train.melt(id_vars=['label_bnc'], 
                          value_vars=['s1', 's2', 's3', 's4'], 
                          var_name='Sensor', 
                          value_name='Unit')

# Création des boîtes à moustaches (boxplots) juxtaposées
fig_box = px.box(df_melted_label, 
                 x='label_bnc', 
                 y='Unit', 
                 facet_col='Sensor',        # Un graphique par colonne pour chaque capteur
                 color='label_bnc',          # Colorer différemment les classes 0 et 1
                 title='Distribution of sensors depending on engine state (0 = Healthy, 1 = Faulty)')

# Indispensable : désolidariser les axes Y car les capteurs ont des échelles très différentes
fig_box.update_yaxes(matches=None)

# Rendre l'axe X catégoriel pour un meilleur affichage
fig_box.update_xaxes(type='category')

fig_box.show()

In [17]:
fig_box.write_image("../figures/sensors_boxplot.jpeg")